### GitHub_NOMAD_Data extraction 

In [ ]:
import requests
import pandas as pd

base_url = 'http://nomad-lab.eu/prod/v1/api/v1'

query = {
    "query": {
        "results.material.elements": {"any": ["W", "Mo", "V", "Cr", "Ti", "Nb"]},
        "results.material.structural_type": "bulk",
        "results.properties.electronic.band_gap.value": {"lte": 0.1},
        "results.material.n_elements": {"lte": 4},
        "results.material.symmetry.bravais_lattice": {"any": ["cI", "cF", "hP"]},
        "results.material.topology.cell.mass_density": {"gte": 5000}
    },
}


def get_all_entries():

    page_size = 500
    search_after = None
    all_entries = []
    first_page = True

    while True:

        pagination = {
            "page_size": page_size,
            "order_by": "entry_id",
            "order": "asc"
        }

        if search_after:
            pagination["search_after"] = [search_after]

        response = requests.post(
            f'{base_url}/entries/query',
            json={
                "query": query["query"],
                "pagination": pagination,
                "required": {"include": ["entry_id"]}
            }
        )

        response_json = response.json()
        entries = response_json.get("data", [])

        if first_page:
            total_entries = response_json.get("pagination", {}).get("total")
            print(f"Total number of matches: {total_entries}")
            first_page = False

        if not entries:
            print("No more entries – end reached.")
            break

        all_entries.extend(entries)

        search_after = entries[-1]["entry_id"]

        print(f"{len(all_entries)} entries loaded...")

        # Stop exactly when total_entries is reached
        if total_entries and len(all_entries) >= total_entries:
            print(f"All {total_entries} entries loaded – finished.")
            break

    all_entry_ids = [e["entry_id"] for e in all_entries]

    print(f"Finished! {len(all_entries)} total entries loaded.")

    return all_entries, all_entry_ids


# Run function
all_entries, all_entry_ids = get_all_entries()

print(f"✅ Total number of entry_ids: {len(all_entry_ids)}")


output_file = "results_all.csv"
batch_size = 500

# Empty DataFrame with predefined columns
columns = [
    "entry_id",
    "is_stable",
    "space_group_symbol",
    "space_group_number",
    "point_group",
    "final_displacement_maximum",
    "crystal_system",
    "bravais_lattice",
    "chemical_formula_reduced",
    "chemical_formula_iupac",
    "mass_fractions",
    "n_temperatures",
    "thermal_expansion",
    "thermal_conductivity",
    "a", "b", "c",
    "alpha", "beta", "gamma",
    "volume",
    "mass_density",
    "n_atoms",
    "n_values",
    "heat_capacity_c_v_specific",
    "n_elements",
    "elements",
    "energy_hulll"
]

df = pd.DataFrame(columns=columns)

print("Starting archive query")

for entry in all_entries:

    entry_id = entry["entry_id"]

    archive_response = requests.post(
        f"{base_url}/entries/{entry_id}/archive/query",
        json={
            "required": {
                "workflow": {
                    "convex_hull": {
                        "energy_hulll": "*"
                    },
                    "thermodynamics": {
                        "n_values": "*",
                        "heat_capacity_c_v_specific": "*",
                        "stability": {
                            "is_stable": "*"
                        }
                    },
                    "debye_model": {
                        "n_temperatures": "*",
                        "thermal_expansion": "*",
                        "thermal_conductivity": "*"
                    }
                },
                "results": {
                    "material": {
                        "n_elements": "*",
                        "elements": "*",
                        "chemical_formula_reduced": "*",
                        "chemical_formula_iupac": "*",
                        "elemental_composition": {
                            "mass_fraction": "*"
                        },
                        "symmetry": {
                            "crystal_system": "*",
                            "bravais_lattice": "*"
                        },
                        "topology": {
                            "n_atoms": "*",
                            "cell": "*",
                            "symmetry": {
                                "space_group_symbol": "*",
                                "space_group_number": "*",
                                "point_group": "*"
                            }
                        }
                    },
                    "properties": {
                        "geometry_optimization": {
                            "final_displacement_maximum": "*"
                        }
                    }
                }
            }
        }
    )

    archive_json = archive_response.json()

    archive_results = archive_json.get("data", {}).get("archive", {}).get("results", {})

    # Extract topology information
    topologies = archive_results.get("material", {}).get("topology", [])

    space_group_symbol = None
    space_group_number = None
    point_group = None

    for topo in topologies:

        sym = topo.get("symmetry")

        # Access space_group_number
        if sym and "space_group_number" in sym:
            space_group_number = sym["space_group_number"]

        # Access space_group_symbol
        if sym and "space_group_symbol" in sym:
            space_group_symbol = sym["space_group_symbol"]

        # Access point_group
        if sym and "point_group" in sym:
            point_group = sym["point_group"]
            break

    # Access thermodynamic stability property
    thermodynamics = archive_json.get("workflow", {}).get("thermodynamics", {})
    is_stable = thermodynamics.get("stability", {}).get("is_stable")

    # Lattice parameters
    topologies = archive_results.get("material", {}).get("topology", [])

    lattice_parameters = {}

    if topologies:
        lattice_parameters = topologies[0].get("cell", {})
        n_atoms = topologies[0].get("n_atoms", {})

    # Extract lattice parameters
    a = lattice_parameters.get("a")
    b = lattice_parameters.get("b")
    c = lattice_parameters.get("c")

    alpha = lattice_parameters.get("alpha")
    beta = lattice_parameters.get("beta")
    gamma = lattice_parameters.get("gamma")

    volume = lattice_parameters.get("volume")
    mass_density = lattice_parameters.get("mass_density")

    # Access geometry optimization displacement property
    final_displacement_maximum = archive_results.get("properties", {}).get("geometry_optimization", {}).get("final_displacement_maximum", {})

    crystal_system = archive_results.get("material", {}).get("symmetry").get("crystal_system", {})

    bravais_lattice = archive_results.get("material", {}).get("symmetry").get("bravais_lattice", {})

    chemical_formula_reduced = archive_results.get("material", {}).get("chemical_formula_reduced", {})

    chemical_formula_iupac = archive_results.get("material", {}).get("chemical_formula_iupac", {})

    n_elements = archive_results.get("material", {}).get("n_elements", {})

    elements = archive_results.get("material", {}).get("elements", {})

    # Extract mass fractions
    elemental_composition = archive_results.get("material", {}).get("elemental_composition", [])

    mass_fractions = [el.get("mass_fraction") for el in elemental_composition]

    # Debye model parameters
    debye_model = archive_results.get("workflow", {}).get("debye_model", [])

    # Temperature points
    n_temperatures = [n_temp.get("n_temperatures") for n_temp in debye_model]

    # Thermal expansion values
    thermal_expansion = [therm_ex.get("thermal_expansion") for therm_ex in debye_model]

    # Thermal conductivity values
    thermal_conductivity = [therm_conduc.get("thermal_conductivity") for therm_conduc in debye_model]

    # Thermodynamic values
    n_values = [n_values.get("n_values") for n_values in thermodynamics]

    heat_capacity_c_v_specific = [
        heat_capacity_c_v_specific.get("heat_capacity_c_v_specific")
        for specific_heat in thermodynamics
    ]

    # Convex hull energy
    energy_hulll = archive_results.get("workflow", {}).get("convex_hull", {}).get("energy_hulll", {})

    new_row = {
        "entry_id": entry_id,
        "is_stable": is_stable,
        "space_group_symbol": space_group_symbol,
        "space_group_number": space_group_number,
        "point_group": point_group,
        "final_displacement_maximum": final_displacement_maximum,
        "crystal_system": crystal_system,
        "bravais_lattice": bravais_lattice,
        "chemical_formula_reduced": chemical_formula_reduced,
        "chemical_formula_iupac": chemical_formula_iupac,
        "mass_fractions": mass_fractions,
        "n_temperatures": n_temperatures,
        "thermal_expansion": thermal_expansion,
        "thermal_conductivity": thermal_conductivity,
        "a": a,
        "b": b,
        "c": c,
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma,
        "volume": volume,
        "mass_density": mass_density,
        "n_atoms": n_atoms,
        "n_values": n_values,
        "heat_capacity_c_v_specific": heat_capacity_c_v_specific,
        "n_elements": len(elements),
        "elements": elements,
        "energy_hulll": energy_hulll
    }

    df.loc[len(df)] = new_row

    # Save intermediate results in batches
    if len(df) % batch_size == 0:
        df.to_csv(output_file, index=False)
        print(f"{len(df)} entries saved.")

print(df)

df.to_csv(output_file, index=False)

print("Final file saved.")